# Coffea NanoEvents — practical walkthrough

This notebook reproduces the core, practical parts of the Coffea NanoEvents tutorial with concise explanations and runnable examples.
Focus: reading NanoAOD, exploring collections (jagged arrays), selections & aggregations, four-vectors, plotting, and chunked/efficient I/O.

What you'll learn:
- How to load NanoAOD into coffea/awkward (NanoEvents)
- Inspect branch/collection structure and work with jagged arrays
- Perform selections, per-event aggregation and vector (4-vector) ops
- Best practices for performance and streaming large files

Prerequisites and notes:
- This notebook requires: coffea, uproot, awkward, vector, matplotlib.
- It attempts to download a small example NanoAOD from the Coffea repo; if that fails, point `fn` to a local NanoAOD file.
- All examples are small and safe to run interactively; replace sample paths with your dataset for real analyses.

In [ ]:
# Install minimal dependencies only if they're missing (safe in notebooks)
try:
    import coffea, uproot, awkward as ak, vector, matplotlib
except Exception:
    import sys
    print('one-time install of coffea/uproot/awkward/vector/matplotlib (may take a minute)')
    !{sys.executable} -m pip install -q "coffea[uproot]" uproot awkward vector matplotlib

# print versions (quick sanity check)
import coffea, uproot, awkward as ak, vector, matplotlib
ak.behavior.update(vector.behavior)  # enable vector methods (p4.mass, delta_r, ...)
print('coffea', getattr(coffea, '__version__', 'unknown'))
print('uproot', uproot.__version__)
print('awkward', ak.__version__)
print('vector', vector.__version__)
print('matplotlib', matplotlib.__version__)

In [ ]:
# core imports and small helpers
import awkward as ak
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import uproot
import numpy as np
import matplotlib.pyplot as plt
import vector

# make sure vector behavior is available for awkward arrays
ak.behavior.update(vector.behavior)
plt.style.use('seaborn-v0_8')

def get_sample_file(dest='nano_dy.root', url=None):
    "Download a tiny NanoAOD sample used for examples (returns local path)."
    import os, urllib.request
    if url is None:
        url = 'https://raw.githubusercontent.com/CoffeaTeam/coffea/main/tests/samples/nano_dy.root'
    if os.path.exists(dest):
        return dest
    try:
        print('downloading example from', url)
        urllib.request.urlretrieve(url, dest)
        return dest
    except Exception as exc:
        raise RuntimeError(f'could not download sample: {exc} -- provide a local NanoAOD and set `fn` accordingly')

In [ ]:
# get sample file (replace `fn` with your file or fileset for real analyses)
fn = get_sample_file()
print('sample file:', fn)

# quick, zero-allocation inspection with uproot
with uproot.open(fn) as f:
    print('top-level keys:', f.keys())
    tree = f['Events']
    print('number of entries:', tree.num_entries)
    print('example branches (first 40):', list(tree.keys())[:40])
    small = tree.arrays(['Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass'], entry_stop=5, library='ak')
    print('shapes (first 5 events):', {k: v.shape for k, v in small.items()})
    print('sample Muon_pt (per-event jagged arrays):
', small['Muon_pt'])

Create NanoEvents and explore structure:
- NanoEvents exposes NanoAOD collections (Muon, Jet, ...) as jagged awkward arrays with convenient physics methods.
- Use NanoEventsFactory.from_root(...) for a quick conversion from a NanoAOD file to a NanoEvents object.

In [ ]:
# load the file into a NanoEvents object (convenience wrapper around uproot+awkward)
events = NanoEventsFactory.from_root(fn, schemaclass=NanoAODSchema).events()

print('type:', type(events))
print('number of events:', len(events))
print('top-level fields (sample):', events.fields[:80])
print('Muon collection fields:', events.Muon.fields)
print('example: muon pt (first 5 events):
', events.Muon.pt[:5])

Working with jagged collections (common patterns):
- ak.num(collection) gives the per-event multiplicity.,
- boolean masks applied to a collection perform elementwise selection,
- ak.max / ak.firsts / ak.topk are useful for per-event reductions.

In [ ]:
# basic selections and counts
muons = events.Muon
print('nMuon per event (first 10):', ak.num(muons)[:10])
# select 'tight' muons (example selection)
tight = muons[(muons.pt > 25) & (abs(muons.eta) < 2.4)]
print('nTight per event (first 10):', ak.num(tight)[:10])
print('fraction of events with >=1 tight muon:', float((ak.num(tight) > 0).mean()))

In [ ]:
# per-event aggregation and plotting (leading muon pt)
leading_pt = ak.max(tight.pt, axis=1)
mask = ak.num(tight) > 0
data = ak.to_numpy(leading_pt[mask])
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.hist(data, bins=50, range=(0,200), histtype='stepfilled', alpha=0.8)
plt.xlabel('leading tight muon pt [GeV]')
plt.ylabel('events')
plt.grid(True)
plt.show()

Working with four-vectors and composite objects:
- Use ak.zip(..., with_name='LorentzVector') (vector integration) to get physics methods like .mass and .delta_r.
- ak.topk / ak.argsort help pick leading objects per event.

In [ ]:
# build 4-vectors for the selected (tight) muons and make a dimuon mass plot
mu_p4 = ak.zip({
    'pt': tight.pt,
    'eta': tight.eta,
    'phi': tight.phi,
    'mass': tight.mass
}, with_name='LorentzVector')
# take the two highest-pt tight muons (per event)
pairs = ak.topk(mu_p4, 2, 'pt')
has_two = ak.num(pairs) == 2
dimu = (pairs[has_two][:,0] + pairs[has_two][:,1]).mass
plt.figure(figsize=(6,4))
plt.hist(ak.to_numpy(dimu), bins=80, range=(0,200))
plt.xlabel('dimuon mass [GeV]')
plt.ylabel('events')
plt.axvspan(80, 100, color='C1', alpha=0.2, label='Z window')
plt.legend()
plt.show()
print('fraction in 80--100 GeV window:', float(((dimu>80)&(dimu<100)).mean()))

In [ ]:
# convert a few per-event scalars into a pandas DataFrame (only for small samples)
import pandas as pd
df = pd.DataFrame({
    'nMuon': ak.to_numpy(ak.num(events.Muon)),
    'nJet': ak.to_numpy(ak.num(events.Jet)),
    'leadingMuon_pt': ak.to_numpy(ak.fill_none(ak.max(events.Muon.pt, axis=1), 0))
})
df.head()

Performance: streaming and working with subsets of branches
- For large datasets do NOT load all branches into memory. Use uproot.iterate or select branch lists.
- Accumulate histograms (numpy/coffea.hist) per-chunk to keep memory small.
- Avoid converting full datasets to pandas; convert only small, flattened summaries.

In [ ]:
# example: chunked processing with uproot.iterate (library='ak' returns awkward arrays)
branches = ['Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass']
bins = np.linspace(0, 200, 101)
acc = np.zeros(len(bins)-1, dtype=np.int64)
for arrays in uproot.iterate(fn, 'Events', branches, step_size=100_000, library='ak'):
    mu = ak.zip({
        'pt': arrays['Muon_pt'],
        'eta': arrays['Muon_eta'],
        'phi': arrays['Muon_phi'],
        'mass': arrays['Muon_mass']
    }, depth_limit=1)
    tight_mu = mu[(mu.pt > 25) & (abs(mu.eta) < 2.4)]
    lead = ak.max(tight_mu.pt, axis=1)
    lead_vals = ak.to_numpy(lead[ak.num(tight_mu) > 0])
    h, _ = np.histogram(lead_vals, bins=bins)
    acc += h
print('accumulated entries (leading tight muon):', int(acc.sum()))
# quick plot
plt.step((bins[:-1]+bins[1:])/2, acc, where='mid')
plt.yscale('log')
plt.xlabel('leading tight muon pt [GeV]')
plt.ylabel('counts')
plt.show()

Troubleshooting & tips:
- If branches are missing: inspect tree.keys() and adapt branch names to your NanoAOD version.
- If vector methods (e.g. .mass) are missing: ensure vector.behavior is attached to awkward.
- For reproducible analyses, define a small fileset and run the same selections in chunked mode.

Exercise — opposite-sign (OS) dimuon and the Z peak:
Compute the invariant mass of opposite-sign muon pairs (all combinations), plot the distribution, and report the fraction of events with an OS pair in the Z window (80–100 GeV).
Hints: use ak.combinations on the muon collection (or on mu_p4), filter by charge, then sum the two Lorentz vectors to get the mass.

In [ ]:
# opposite-sign dimuon (all combinations)
mu = events.Muon
tight = mu[(mu.pt > 25) & (abs(mu.eta) < 2.4)]
# include charge when building the LorentzVector so we can filter pairs by sign
mu_p4 = ak.zip({
    'pt': tight.pt,
    'eta': tight.eta,
    'phi': tight.phi,
    'mass': tight.mass,
    'charge': tight.charge
}, with_name='LorentzVector')
pairs = ak.combinations(mu_p4, 2)
os = pairs[(pairs['0'].charge != pairs['1'].charge)]
# masses is a jagged array (0..N pairs per event); flatten for a per-pair distribution
masses = ak.flatten((os['0'] + os['1']).mass)
masses_np = ak.to_numpy(masses)
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.hist(masses_np, bins=80, range=(0,200), histtype='stepfilled', alpha=0.8)
plt.xlabel('opposite-sign dimuon mass [GeV]')
plt.ylabel('pairs')
plt.axvspan(80, 100, color='C1', alpha=0.2, label='Z window')
plt.legend()
plt.show()
# fraction of EVENTS with at least one OS pair in the Z window
has_z = ak.sum(((os['0'] + os['1']).mass > 80) & ((os['0'] + os['1']).mass < 100), axis=1) > 0
print('fraction of events with an OS dimuon in 80-100 GeV:', float(has_z.mean()))

Mini example — porting to coffea.processor (scales to many files):
This is a minimal Processor that fills two histograms (leading muon pt and OS dimuon mass).
Run this on a small fileset to validate logic, then scale up with more workers or HTCondor/Dask executors.

In [ ]:
from coffea import processor, hist

class DimuonProcessor(processor.ProcessorABC):
    def __init__(self):
        self._accumulator = processor.dict_accumulator({
            'lead_mu_pt': hist.Hist('Events', hist.Bin('pt', 'leading mu pt [GeV]', 50, 0, 200)),
            'dimu_mass': hist.Hist('Events', hist.Bin('mass', 'dimu mass [GeV]', 80, 0, 200)),
        })
    @property
    def accumulator(self):
        return self._accumulator
    def process(self, events):
        out = self.accumulator.copy()
        mu = events.Muon
        tight = mu[(mu.pt > 25) & (abs(mu.eta) < 2.4)]
        lead = ak.max(tight.pt, axis=1)
        out['lead_mu_pt'].fill(pt=ak.to_numpy(ak.fill_none(lead, 0)))
        mu_p4 = ak.zip({
            'pt': tight.pt, 'eta': tight.eta, 'phi': tight.phi, 'mass': tight.mass, 'charge': tight.charge
        }, with_name='LorentzVector')
        pairs = ak.combinations(mu_p4, 2)
        os = pairs[(pairs['0'].charge != pairs['1'].charge)]
        masses = ak.flatten((os['0'] + os['1']).mass)
        out['dimu_mass'].fill(mass=ak.to_numpy(masses))
        return out
    def postprocess(self, accumulator):
        return accumulator

# run locally on the single-sample fileset (workers=1 to keep resource usage small)
fileset = {'sample': [fn]}
result = processor.run_uproot_job(fileset, 'Events', DimuonProcessor(), processor.futures_executor, {"schema": NanoAODSchema, "workers": 1}, chunksize=100000)
print('processor output keys:', list(result.keys()))

Quick checklist & common pitfalls:
- Start by running the analysis on a single small file and validate selections visually.
- Use uproot.iterate/select branches to limit memory when scaling.
- Remember to attach vector.behavior to awkward to get .mass/.delta_r methods.
- When porting to coffea.processor, make the process() pure (no global state) and test with a small fileset.